# NB19: Multi-Benchmark Evaluation

**Aim:** Evaluate ModernFinBERT and baseline models on multiple financial sentiment benchmarks to test generalization beyond FPB.

**Benchmarks:**
1. FPB sentences_50agree (4,846 press-release sentences)
2. FPB sentences_allagree (2,264 unanimous-agreement sentences)
3. Twitter Financial News Sentiment (social media)
4. FiQA 2018 Task 1 (financial opinions, continuous→3-class)

**Models:**
1. `neoyipeng/ModernFinBERT-base` (production, max_length=512)
2. `ProsusAI/finbert` (in-domain FinBERT baseline)
3. `yiyanghkust/finbert-tone` (analyst-report FinBERT baseline)

**Runtime:** ~1 hour on T4 (evaluation only, no training)

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn matplotlib seaborn transformers

In [ ]:
import json
import gc
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer

LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
os.makedirs("../results", exist_ok=True)

## 1. Benchmark Loading & Label Verification

Each benchmark has a different label scheme. We standardize to {0: NEGATIVE, 1: NEUTRAL, 2: POSITIVE}.

In [ ]:
def load_fpb_50agree():
    ds = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
    return list(ds["sentence"]), list(ds["label"])  # 0=neg, 1=neu, 2=pos

def load_fpb_allagree():
    ds = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
    return list(ds["sentence"]), list(ds["label"])

def load_twitter_financial():
    ds = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation")
    texts = list(ds["text"])
    raw_labels = list(ds["label"])
    # Inspect label scheme: check id2label or sample data
    # zeroshot/twitter-financial-news-sentiment: 0=Bearish, 1=Bullish, 2=Neutral
    remap = {0: 0, 1: 2, 2: 1}  # Bearish→NEG, Bullish→POS, Neutral→NEU
    labels = [remap[l] for l in raw_labels]
    return texts, labels

def load_fiqa(lo=-0.2, hi=0.2):
    ds = load_dataset("pauri32/fiqa-2018", split="train")
    texts, labels = [], []
    for row in ds:
        score = row["sentiment_score"]
        if score < lo:
            labels.append(0)
        elif score > hi:
            labels.append(2)
        else:
            labels.append(1)
        texts.append(row["sentence"])
    return texts, labels

BENCHMARKS = {
    "fpb_50agree": {"loader": load_fpb_50agree, "desc": "FPB 50% agree (4,846)"},
    "fpb_allagree": {"loader": load_fpb_allagree, "desc": "FPB 100% agree (2,264)"},
    "twitter_fin": {"loader": load_twitter_financial, "desc": "Twitter Financial News Sentiment"},
    "fiqa_2018": {"loader": lambda: load_fiqa(-0.2, 0.2), "desc": "FiQA 2018 (thresholds ±0.2)"},
}

print("Loading and verifying benchmarks...\n")
benchmark_cache = {}
for name, cfg in BENCHMARKS.items():
    texts, labels = cfg["loader"]()
    benchmark_cache[name] = (texts, labels)
    dist = {LABEL_NAMES[i]: labels.count(i) for i in range(3)}
    print(f"{name} ({cfg['desc']}): {len(texts)} samples")
    print(f"  Distribution: {dist}")
    print(f"  Sample NEG: {texts[labels.index(0)][:80]}..." if 0 in labels else "  No NEG samples")
    print(f"  Sample POS: {texts[labels.index(2)][:80]}..." if 2 in labels else "  No POS samples")
    print()

## 2. Model Definitions & Label Remapping

In [ ]:
MODELS = {
    "ModernFinBERT": {
        "model_id": "neoyipeng/ModernFinBERT-base",
        "max_length": 512,
        "label_remap": None,
    },
    "ProsusAI/finbert": {
        "model_id": "ProsusAI/finbert",
        "max_length": 512,
        "label_remap": None,  # will be determined from id2label
    },
    "finbert-tone": {
        "model_id": "yiyanghkust/finbert-tone",
        "max_length": 512,
        "label_remap": None,  # will be determined from id2label
    },
}

# Detect label remapping from model config
TARGET_SCHEME = {"negative": 0, "neutral": 1, "positive": 2}

for model_name, cfg in MODELS.items():
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"])
    model = AutoModelForSequenceClassification.from_pretrained(cfg["model_id"])
    id2label = model.config.id2label
    print(f"{model_name}: id2label = {id2label}")

    remap = {}
    for idx, label_str in id2label.items():
        normalized = label_str.lower().strip()
        if normalized in TARGET_SCHEME:
            remap[int(idx)] = TARGET_SCHEME[normalized]
        elif "neg" in normalized or "bear" in normalized:
            remap[int(idx)] = 0
        elif "neu" in normalized:
            remap[int(idx)] = 1
        elif "pos" in normalized or "bull" in normalized:
            remap[int(idx)] = 2
        else:
            print(f"  WARNING: unknown label '{label_str}' at index {idx}")
            remap[int(idx)] = int(idx)

    is_identity = all(k == v for k, v in remap.items())
    cfg["label_remap"] = None if is_identity else remap
    print(f"  Remap: {'identity (none needed)' if is_identity else remap}")

    del model, tokenizer
    gc.collect()

## 3. Inference & Evaluation Functions

In [ ]:
def run_inference(model, tokenizer, texts, max_length=512, label_remap=None, batch_size=32):
    device = next(model.parameters()).device
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True,
                               truncation=True, max_length=max_length)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            if label_remap:
                preds = np.array([label_remap[p] for p in preds])
            all_preds.extend(preds)
    return np.array(all_preds)


def evaluate_on_benchmark(model, tokenizer, texts, labels, max_length=512, label_remap=None):
    preds = run_inference(model, tokenizer, texts, max_length, label_remap)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    report = classification_report(labels, preds, target_names=LABEL_NAMES,
                                    output_dict=True, zero_division=0)
    per_class = {cls: round(report[cls]["f1-score"], 4) for cls in LABEL_NAMES}
    return {"accuracy": round(acc, 4), "macro_f1": round(f1, 4), "per_class_f1": per_class, "n": len(texts)}

## 4. Sanity Check: Verify Label Remapping

Run each model on a few obvious samples to confirm labels are correctly mapped.

In [ ]:
sanity_texts = [
    "The company reported a massive loss and is laying off 40% of staff.",     # obvious NEG
    "Revenue was flat compared to last quarter.",                                # obvious NEU
    "Earnings beat expectations by 20%, stock surged to all-time high.",        # obvious POS
]
sanity_expected = [0, 1, 2]

# CPU fallback for P100 GPU compatibility (like NB16)
device = torch.device("cpu")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap[0] >= 7:
        device = torch.device("cuda")
    else:
        print(f"GPU {torch.cuda.get_device_name()} (sm_{cap[0]}{cap[1]}) incompatible — using CPU")
print(f"Device: {device}")

for model_name, cfg in MODELS.items():
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"])
    model = AutoModelForSequenceClassification.from_pretrained(cfg["model_id"])
    model = model.to(device).eval()

    preds = run_inference(model, tokenizer, sanity_texts, cfg["max_length"], cfg["label_remap"])
    correct = sum(p == e for p, e in zip(preds, sanity_expected))
    status = "PASS" if correct == 3 else "CHECK"
    print(f"{model_name}: {correct}/3 correct [{status}]")
    for text, pred, exp in zip(sanity_texts, preds, sanity_expected):
        match = "ok" if pred == exp else "MISMATCH"
        print(f"  {LABEL_NAMES[pred]:>8} (exp {LABEL_NAMES[exp]:>8}) [{match}] {text[:60]}")

    del model, tokenizer
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

## 5. Run All Evaluations

In [ ]:
all_results = {}

for model_name, cfg in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"])
    model = AutoModelForSequenceClassification.from_pretrained(cfg["model_id"])
    model = model.to(device).eval()

    model_results = {}
    for bench_name, (texts, labels) in benchmark_cache.items():
        result = evaluate_on_benchmark(
            model, tokenizer, texts, labels,
            max_length=cfg["max_length"], label_remap=cfg["label_remap"],
        )
        model_results[bench_name] = result
        print(f"  {bench_name:20s}: acc={result['accuracy']:.4f}, f1={result['macro_f1']:.4f} (n={result['n']})")

    all_results[model_name] = model_results
    del model, tokenizer
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

with open("../results/multi_benchmark_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nSaved to results/multi_benchmark_results.json")

## 6. FiQA Threshold Sensitivity

In [ ]:
FIQA_THRESHOLDS = [(-0.1, 0.1), (-0.2, 0.2), (-0.3, 0.3)]

tokenizer = AutoTokenizer.from_pretrained("neoyipeng/ModernFinBERT-base")
model = AutoModelForSequenceClassification.from_pretrained("neoyipeng/ModernFinBERT-base")
model = model.to(device).eval()

fiqa_sensitivity = []
for lo, hi in FIQA_THRESHOLDS:
    texts, labels = load_fiqa(lo, hi)
    preds = run_inference(model, tokenizer, texts, max_length=512)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    dist = {LABEL_NAMES[i]: labels.count(i) for i in range(3)}
    fiqa_sensitivity.append({"thresholds": f"({lo}, {hi})", "accuracy": round(acc, 4),
                              "macro_f1": round(f1, 4), "distribution": dist, "n": len(texts)})
    print(f"  FiQA ({lo}, {hi}): acc={acc:.4f}, f1={f1:.4f}, n={len(texts)}, dist={dist}")

del model, tokenizer
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

## 7. Results Tables

In [ ]:
rows = []
for model_name, benchmarks in all_results.items():
    row = {"Model": model_name}
    for bench_name, metrics in benchmarks.items():
        row[f"{bench_name}_acc"] = metrics["accuracy"]
        row[f"{bench_name}_f1"] = metrics["macro_f1"]
    rows.append(row)

results_df = pd.DataFrame(rows)

print("=" * 100)
print("MULTI-BENCHMARK COMPARISON")
print("=" * 100)
print(results_df.to_string(index=False, float_format="%.4f"))

# Identify where ModernFinBERT wins/loses
print("\n\nModernFinBERT vs baselines:")
mfb = all_results.get("ModernFinBERT", {})
for model_name in ["ProsusAI/finbert", "finbert-tone"]:
    baseline = all_results.get(model_name, {})
    for bench in benchmark_cache:
        if bench in mfb and bench in baseline:
            delta = mfb[bench]["accuracy"] - baseline[bench]["accuracy"]
            marker = "WIN" if delta > 0 else "LOSE" if delta < 0 else "TIE"
            print(f"  vs {model_name} on {bench}: {delta*100:+.1f}pp [{marker}]")

## 8. LaTeX Table for Paper

In [ ]:
bench_names = list(benchmark_cache.keys())
bench_short = {"fpb_50agree": "FPB-50", "fpb_allagree": "FPB-All",
               "twitter_fin": "Twitter", "fiqa_2018": "FiQA"}

print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Multi-benchmark evaluation. ModernFinBERT trained on aggregated data "
      r"(FPB held out for FPB benchmarks). ProsusAI/finbert trained on FPB (in-domain). "
      r"finbert-tone trained on analyst reports (zero-shot on FPB).}")
print(r"\label{tab:multi-benchmark}")

cols = "l" + "cc" * len(bench_names)
print(f"\\begin{{tabular}}{{{cols}}}")
print(r"\toprule")

header = r"\textbf{Model}"
for bn in bench_names:
    short = bench_short.get(bn, bn).replace("_", r"\_")
    header += f" & \\multicolumn{{2}}{{c}}{{\\textbf{{{short}}}}}"
header += r" \\"
print(header)

cmidlines = ""
for i in range(len(bench_names)):
    start = 2 + i * 2
    cmidlines += f"\\cmidrule(lr){{{start}-{start+1}}} "
print(cmidlines)

sub = ""
for _ in bench_names:
    sub += r" & Acc & F1"
sub += r" \\"
print(sub)
print(r"\midrule")

for model_name in all_results:
    line = model_name.replace("_", r"\_")
    for bn in bench_names:
        m = all_results[model_name].get(bn, {})
        acc = m.get("accuracy", "—")
        f1_val = m.get("macro_f1", "—")
        if isinstance(acc, float):
            line += f" & {acc:.4f} & {f1_val:.4f}"
        else:
            line += f" & {acc} & {f1_val}"
    line += r" \\"
    print(line)

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

## 9. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
model_names = list(all_results.keys())
colors = ["#4CAF50", "#2196F3", "#FF9800"]

for ax, metric, title in zip(axes, ["accuracy", "macro_f1"], ["Accuracy", "Macro F1"]):
    x = np.arange(len(bench_names))
    width = 0.25

    for i, mn in enumerate(model_names):
        vals = [all_results[mn].get(bn, {}).get(metric, 0) for bn in bench_names]
        ax.bar(x + i * width, vals, width, label=mn, color=colors[i])

    ax.set_xticks(x + width)
    ax.set_xticklabels([bench_short.get(bn, bn) for bn in bench_names])
    ax.set_ylabel(title)
    ax.set_title(f"Multi-Benchmark {title}")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("multi_benchmark_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
mfb = all_results.get("ModernFinBERT", {})
wins = sum(1 for bn in bench_names
           if all(mfb.get(bn, {}).get("accuracy", 0) >= all_results[other].get(bn, {}).get("accuracy", 1)
                  for other in all_results if other != "ModernFinBERT"))

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"ModernFinBERT wins on {wins}/{len(bench_names)} benchmarks (by accuracy)")
print(f"\nFiQA threshold sensitivity (ModernFinBERT):")
for s in fiqa_sensitivity:
    print(f"  {s['thresholds']}: acc={s['accuracy']}, f1={s['macro_f1']}")